# Idiom-Aware Fine-Tuning of EN-DE Translations

Idea: 2-stage fine-tuning of a pre-trained model to increase idiom translation accuracy without sacrificing general language translation quality

Specifically in this notebook I want to focus on Parameter Efficient Fine-tuning (PEFT) techniques.

---
**Stages**

---



**Stage 0**: Baseline (Helsikin-NLP model)

**Stage 1**: Immediate Idiom-only fine-tuning
- Increase the model's idiom translation performance by fine-tuning the baseline from stage 0 on EN-DE idiom sentence pairs (`IdiomsInCtx-MT`)
- This will likely increase idiom-specific translation performance (measured as `Idiom BLEU`), and likely decrease general language translation (measured as `WMT BLEU)` performance.

**Stage 2**: General Language fine-tuning
- Fine-tune the model from stage 1 on a general language EN-DE dataset (`WMT-14`)
- The hope is to maintain the learned idiom-translation ability from stage 1, while recovering some of the lost general language translation perfromance.

Ideally, our stage 2 model has not sacrificied its general language translation capabilties, but has notably increased idiom-specific translation capabilities.

---
**Experiments** in this notebook

---
- **Stage 1:** Idioms fine-tuning (LoRA)
- **Stage 2:** WMT fine-tuning (LoRA) with frozen encoder

With ranked decoding sweep {4,8,16} with
- greedy
- beam search with num_beams = {4,8}
- length_penalty = {0.8, 1.0, 1.2)
- early_stopping

## Clone Repo from Git / Commit Changes
Run the git clone everytime a new session is started

In [1]:
!rm -rf Idiom-Aware-Finetuning-in-MT-EN-to-DE
!git clone https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
%cd Idiom-Aware-Finetuning-in-MT-EN-to-DE
!ls

Cloning into 'Idiom-Aware-Finetuning-in-MT-EN-to-DE'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 86 (delta 36), reused 34 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 215.01 KiB | 14.33 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/Idiom-Aware-Finetuning-in-MT-EN-to-DE
configs  notebooks  README.md  results	scripts  src


Copy notebook from Drive into Repo

In [2]:
# !cp "/content/drive/MyDrive/ds266_idiom_mt/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb" notebooks/

Add and commit changes

In [3]:
#!git config --global user.email "auntiepersilla@gmail.com"
#!git config --global user.name "Michael Strommer"

#!git add notebooks/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb
#!git commit -m "Add all-in-one Drive-first pipeline notebook"

Push to GitHub

In [4]:
#!git remote set-url origin https://auntiepersilla:<TOKEN>@github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
#!git push origin main

## Mount Drive, Create Paths

In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
CACHE_DIR = os.path.join(DRIVE_ROOT, "cache")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR

print("DRIVE_ROOT:", DRIVE_ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("CACHE_DIR:", CACHE_DIR)

Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/ds266_idiom_mt
CHECKPOINT_DIR: /content/drive/MyDrive/ds266_idiom_mt/checkpoints
RESULTS_DIR: /content/drive/MyDrive/ds266_idiom_mt/results
CACHE_DIR: /content/drive/MyDrive/ds266_idiom_mt/cache


## Install Dependencies & Imports

In [6]:
!pip -q install -U "transformers>=4.38" datasets evaluate sacrebleu accelerate sentencepiece sacremoses "protobuf<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.6 MB/s eta 0:00:00


In [7]:
import random, numpy as np, torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
import sacrebleu
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Reproducibility / rerun controls
FORCE_RETRAIN = False
CANONICAL_DECODE = "greedy"

def checkpoint_exists(model_dir):
    if not os.path.isdir(model_dir):
        return False
    has_config = os.path.exists(os.path.join(model_dir, "adapter_config.json")) or os.path.exists(os.path.join(model_dir, "config.json"))
    has_weights = (
        os.path.exists(os.path.join(model_dir, "adapter_model.bin")) or
        os.path.exists(os.path.join(model_dir, "adapter_model.safetensors")) or
        os.path.exists(os.path.join(model_dir, "pytorch_model.bin")) or
        os.path.exists(os.path.join(model_dir, "model.safetensors"))
    )
    return has_config and has_weights

def clear_generation_max_length(model):
    if hasattr(model, "generation_config") and model.generation_config is not None:
        try:
            model.generation_config.max_length = None
        except Exception:
            pass
    if hasattr(model, "config") and model.config is not None:
        try:
            model.config.max_length = None
        except Exception:
            pass
    return model


device: cuda


In [8]:
!pip -q install peft

In [9]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch.nn as nn

## Load Data

- Idiom-only Data: davidstap/IdiomsInCtx-MT (1000 EN-DE idiom sentence pairs)
  - train: 800
  - dev: 100
  - test: 100
- General Translation Training and Test Data: WMT-14 EN-DE
  - train: 20.000
  - test: 2.000
  - test: 3.003

In [10]:
def load_idioms(name="davidstap/IdiomsInCtx-MT", config="en-de", train_frac=0.8, dev_frac=0.1, seed=42):
    raw = load_dataset(name, config)

    full = raw["test"] if ("test" in raw and len(raw.keys()) == 1) else raw[list(raw.keys())[0]]
    src_lang, tgt_lang = config.split("-")

    def standardize(ex):
        if "translation" in ex and src_lang in ex["translation"] and tgt_lang in ex["translation"]:
            return {"src": ex["translation"][src_lang], "tgt": ex["translation"][tgt_lang]}
        if src_lang in ex and tgt_lang in ex:
            return {"src": ex[src_lang], "tgt": ex[tgt_lang]}
        raise ValueError(list(ex.keys()))

    full = full.map(standardize)
    full = full.remove_columns([c for c in full.column_names if c not in ["src","tgt"]])

    tmp = full.train_test_split(test_size=(1-train_frac), seed=seed)
    train, rest = tmp["train"], tmp["test"]

    dev_frac_of_rest = dev_frac / (1-train_frac)
    tmp2 = rest.train_test_split(test_size=(1-dev_frac_of_rest), seed=seed)
    dev, test = tmp2["train"], tmp2["test"]

    return DatasetDict({"train": train, "dev": dev, "test": test})

def load_wmt14(ft_train_n=20000, ft_dev_n=2000, seed=42):
    wmt = load_dataset("wmt14", "de-en")
    def to_en_de(ex):
        tr = ex["translation"]
        return {"src": tr["en"], "tgt": tr["de"]}

    train = wmt["train"].map(to_en_de, remove_columns=wmt["train"].column_names)
    test = wmt["test"].map(to_en_de, remove_columns=wmt["test"].column_names)

    shuf = train.shuffle(seed=seed)
    ft_train = shuf.select(range(min(ft_train_n, len(shuf))))
    ft_dev = shuf.select(range(min(ft_train_n, len(shuf)), min(ft_train_n+ft_dev_n, len(shuf))))
    return DatasetDict({"ft_train": ft_train, "ft_dev": ft_dev, "wmt_test": test})

idiom_ds = load_idioms(seed=SEED)
general_ds = load_wmt14(seed=SEED)

print("idiom_ds:", {k: len(v) for k,v in idiom_ds.items()})
print("general_ds:", {k: len(v) for k,v in general_ds.items()})
print("idiom sample:", idiom_ds["test"][0])

idiom_ds: {'train': 800, 'dev': 100, 'test': 100}
general_ds: {'ft_train': 20000, 'ft_dev': 2000, 'wmt_test': 3003}
idiom sample: {'src': 'Not a word of all this to anyone!', 'tgt': 'Kein Wort von all dem zu irgendjemandem!'}


## Metrics and Generation Helpers

In [11]:
# Set Global Hyperparameters for Predictions
max_source_len = 256
max_new_tokens = 128
batch_size = 16

In [12]:
@torch.no_grad()
# Generate Predictions
def generate_preds(model, tok, sources, max_source_len=max_source_len, max_new_tokens=max_new_tokens, batch_size=batch_size, gen_kwargs=None):
    model.eval()
    gen_kwargs = gen_kwargs or {}
    preds=[]
    import copy
    gen_cfg = copy.deepcopy(model.generation_config) if hasattr(model, "generation_config") else None
    if gen_cfg is not None:
        try:
            gen_cfg.max_length = None
        except Exception:
            pass
    for i in range(0, len(sources), batch_size):
        batch = sources[i:i+batch_size]
        inputs = tok(batch, return_tensors="pt", truncation=True, padding=True, max_length=max_source_len).to(device)
        if gen_cfg is not None:
            out = model.generate(**inputs, generation_config=gen_cfg, max_new_tokens=max_new_tokens, **gen_kwargs)
        else:
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

DECODING_SETUPS = {
    "greedy": {},
    "beam4": {"num_beams": 4, "early_stopping": True},
    "beam8": {"num_beams": 8, "early_stopping": True, "length_penalty": 1.0},
}

# Compute Metrics
def compute_metrics(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"bleu": float(bleu), "chrf": float(chrf)}

In [13]:
import re
# Metrics Logger

METRICS_PATH = os.path.join(RESULTS_DIR, "metrics.csv")

METRIC_FIELDS = [
    "timestamp",
    "run_name",
    "split",
    "bleu",
    "chrf",
    "model_name",
    "learning_rate",
    "epochs",
    "batch_size",
    "max_source_len",
    "max_new_tokens",
    "freeze_encoder",
    "train_size",
    "dev_size",
    "seed",
    "n_eval",
    "notes",
]

def log_metrics(
    run_name,
    split,
    metrics,
    *,
    model_name=None,
    learning_rate=None,
    epochs=None,
    batch_size=None,
    max_source_len=None,
    max_new_tokens=None,
    freeze_encoder=None,
    train_size=None,
    dev_size=None,
    seed=None,
    n_eval=None,
    notes=None,
):
    # Initialize full schema with None
    row = {k: None for k in METRIC_FIELDS}

    row.update({
        "timestamp": datetime.now(ZoneInfo('America/Los_Angeles')).isoformat(timespec="seconds"),
        "run_name": run_name,
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "model_name": model_name,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "batch_size": batch_size,
        "max_source_len": max_source_len,
        "max_new_tokens": max_new_tokens,
        "freeze_encoder": freeze_encoder,
        "train_size": train_size,
        "dev_size": dev_size,
        "seed": seed,
        "n_eval": n_eval,
        "notes": notes,
    })

    df = pd.DataFrame([row])

    if os.path.exists(METRICS_PATH):
        df.to_csv(METRICS_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(METRICS_PATH, index=False)

    return row

# Canonical automatic-metrics file used downstream for clean analysis merges
CANONICAL_METRICS_PATH = os.path.join(RESULTS_DIR, "automatic_metrics_canonical.csv")

CANONICAL_FIELDS = [
    "timestamp",
    "model",
    "model_label",
    "split",
    "bleu",
    "chrf",
    "source_notebook",
    "source_run_name",
    "notes",
]

def infer_model_label(model, source_run_name=""):
    model_str = str(model)
    run_str = str(source_run_name)
    m = re.search(r"lora_r(4|8|16)", model_str) or re.search(r"lora_r(4|8|16)", run_str)
    if m:
        return f"lora_r{m.group(1)}_stage2"
    return model_str

def upsert_canonical_metric(model, split, metrics, *, source_notebook, source_run_name, notes=""):
    row = {
        "timestamp": datetime.now(ZoneInfo('America/Los_Angeles')).isoformat(timespec="seconds"),
        "model": model,
        "model_label": infer_model_label(model, source_run_name),
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "source_notebook": source_notebook,
        "source_run_name": source_run_name,
        "notes": notes,
    }

    df_new = pd.DataFrame([row], columns=CANONICAL_FIELDS)

    if os.path.exists(CANONICAL_METRICS_PATH):
        df_old = pd.read_csv(CANONICAL_METRICS_PATH)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new

    df["timestamp"] = df["timestamp"].astype(str)
    df = df.sort_values(["model_label", "split", "timestamp"])
    df = df.drop_duplicates(subset=["model_label", "split"], keep="last")
    df.to_csv(CANONICAL_METRICS_PATH, index=False)
    return df_new


### Canonical automatic-metrics writes

This notebook still appends to `results/metrics.csv` as a historical log, but it also upserts clean canonical rows to `results/automatic_metrics_canonical.csv`. For notebook 04, the canonical models owned by this notebook are:

- `lora_r4_stage2`
- `lora_r8_stage2`
- `lora_r16_stage2`

Because this notebook evaluates a decoding sweep, only the canonical decode specified by `CANONICAL_DECODE` is written to the canonical file.

# Stage 0: Baseline Evaluation
- Model: Helsinki-NLP/opus-mt-en-de

In [14]:
# Load Model and Tokenizer
BASE_MODEL = "Helsinki-NLP/opus-mt-en-de"
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
model = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device))

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [15]:
#model.config

In [16]:
# Set test data
idiom_test = idiom_ds["test"]
wmt_test = general_ds["wmt_test"]

# Global Prediction Hyperparameters
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128
BATCH_SIZE = 16

## Baseline Evaluation

In [17]:
# Evaluate Idiom Translation
idiom_pred = generate_preds(model, tok, idiom_test["src"],
                            max_source_len=MAX_SOURCE_LEN,
                            max_new_tokens=MAX_NEW_TOKENS,
                            batch_size=BATCH_SIZE)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("baseline idioms:", idiom_m)

log_metrics(
    "baseline", "idioms_test", idiom_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(idiom_ds["train"]),
    dev_size=len(idiom_ds["dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes="pretrained baseline",
)

# Evaluate General Translation
wmt_pred = generate_preds(model, tok, wmt_test["src"],
                          max_source_len=MAX_SOURCE_LEN,
                          max_new_tokens=MAX_NEW_TOKENS,
                          batch_size=BATCH_SIZE)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("baseline wmt:", wmt_m)

log_metrics(
    "baseline", "wmt_test", wmt_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes="pretrained baseline (WMT test full 3003)",
)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


baseline idioms: {'bleu': 39.64935194574556, 'chrf': 60.786111999956105}
baseline wmt: {'bleu': 27.567994028871393, 'chrf': 58.43184557779851}


{'timestamp': '2026-03-28T21:40:42-07:00',
 'run_name': 'baseline',
 'split': 'wmt_test',
 'bleu': 27.567994028871393,
 'chrf': 58.43184557779851,
 'model_name': 'Helsinki-NLP/opus-mt-en-de',
 'learning_rate': None,
 'epochs': None,
 'batch_size': 16,
 'max_source_len': 256,
 'max_new_tokens': 128,
 'freeze_encoder': False,
 'train_size': 20000,
 'dev_size': 2000,
 'seed': 42,
 'n_eval': 3003,
 'notes': 'pretrained baseline (WMT test full 3003)'}

## Stage 1: Idiom Fine-tuning

In [18]:
def fine_tune(model_name_or_path, train_ds, dev_ds, out_dir, lr, epochs, batch_size=8, freeze_encoder=False):
    tok = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
    model = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path).to(device))

    if freeze_encoder and hasattr(model, "model") and hasattr(model.model, "encoder"):
        for p in model.model.encoder.parameters():
            p.requires_grad = False
        if hasattr(model.model, "shared"):
            for p in model.model.shared.parameters():
                p.requires_grad = False

    def tokenize(batch):
        x = tok(batch["src"], truncation=True, max_length=256)
        y = tok(text_target=batch["tgt"], truncation=True, max_length=256)
        x["labels"] = y["input_ids"]
        return x

    train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
    dev_tok = dev_ds.map(tokenize, batched=True, remove_columns=dev_ds.column_names)

    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
    )

    collator = DataCollatorForSeq2Seq(tok, model=model)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=dev_tok, data_collator=collator)
    trainer.train()

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    return out_dir

In [19]:
# Verify LoRA works with Helsinki NLP
def find_lora_linear_module_names(model, include_substring="decoder", endswith=("q_proj","k_proj","v_proj","out_proj")):
    """Return full module names of Linear layers in (e.g.) decoder that end with attention proj names."""
    names = []
    for name, module in model.named_modules():
        if include_substring in name and name.endswith(endswith) and isinstance(module, nn.Linear):
            names.append(name)
    return names

def load_lora_for_eval(base_model_name_or_path, adapter_dir):
    """Load BASE model + LoRA adapter for inference."""
    base = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(base_model_name_or_path).to(device))
    model = clear_generation_max_length(PeftModel.from_pretrained(base, adapter_dir).to(device))
    tok = AutoTokenizer.from_pretrained(adapter_dir, use_fast=False)
    return model, tok

In [20]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType

def fine_tune_lora(
    base_model_name_or_path,
    train_ds,
    dev_ds,
    out_dir,
    lr,
    epochs,
    batch_size=8,
    freeze_encoder=True,
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    early_stopping_patience=None,
    weight_decay=0.0,
    lora_decoder_only=True,
    adapter_path=None,
    max_source_len=256,
    max_target_len=256,
):
    # Tokenizer + Base model ---
    tok = AutoTokenizer.from_pretrained(base_model_name_or_path, use_fast=False)
    base = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(base_model_name_or_path).to(device))

    # Encoder Freezing option
    if freeze_encoder and hasattr(base, "model") and hasattr(base.model, "encoder"):
        for p in base.model.encoder.parameters():
            p.requires_grad = False
        if hasattr(base.model, "shared"):
            for p in base.model.shared.parameters():
                p.requires_grad = False

    # Attach / load LoRA
    if adapter_path is not None:
        # continue from existing adapter
        model = PeftModel.from_pretrained(base, adapter_path).to(device)
        model.train()
        try:
            model.set_adapter("default")
        except Exception:
            pass

        # Freeze everything and unfreeze only LoRA params (robust)
        for n, p in model.named_parameters():
            p.requires_grad = False
        for n, p in model.named_parameters():
            if "lora_" in n:
                p.requires_grad = True

    else:
        # create a new adapter
        if lora_decoder_only:
            target_modules = find_lora_linear_module_names(base, include_substring="decoder")
            if len(target_modules) == 0:
                raise ValueError("No decoder Linear target modules found for LoRA. Inspect model.named_modules().")
        else:
            target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"]

        lora_cfg = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=target_modules,
            bias="none",
        )
        model = get_peft_model(base, lora_cfg).to(device)

    model.print_trainable_parameters()

    # Tokenize datasets
    def tokenize(batch):
        x = tok(batch["src"], truncation=True, max_length=max_source_len)
        y = tok(text_target=batch["tgt"], truncation=True, max_length=max_target_len)
        x["labels"] = y["input_ids"]
        return x

    train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
    dev_tok   = dev_ds.map(tokenize, batched=True, remove_columns=dev_ds.column_names)

    collator = DataCollatorForSeq2Seq(tok, model=model)

    # Training args
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        weight_decay=weight_decay,
    )

    callbacks = []
    if early_stopping_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    # Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=collator,
        callbacks=callbacks,
    )

    trainer.train()

    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)

    return out_dir

### Run Idiom-only Fine-Tuning

In [21]:
IDIOM_ONLY_DIR = os.path.join(CHECKPOINT_DIR, "idiom_only_v1")

# Set Fine-tuning Hyperparameters
lr = 5e-5
epochs = 3
batch_size = 8

# Run Fine-tuning or reuse checkpoint
if checkpoint_exists(IDIOM_ONLY_DIR) and not FORCE_RETRAIN:
    print(f"Loading existing idiom-only checkpoint from: {IDIOM_ONLY_DIR}")
else:
    print(f"Training idiom-only model and saving to: {IDIOM_ONLY_DIR}")
    fine_tune(BASE_MODEL, idiom_ds["train"], idiom_ds["dev"], IDIOM_ONLY_DIR, lr=lr, epochs=epochs, batch_size=batch_size)

idiom_model = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(IDIOM_ONLY_DIR).to(device))
idiom_tok = AutoTokenizer.from_pretrained(IDIOM_ONLY_DIR, use_fast=False)

Loading existing idiom-only checkpoint from: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/idiom_only_v1


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## Stage 2: PEFT/LoRA

Decoding with ranks {4,8,16)

In [22]:
LORA_RANKS = [4, 8, 16]

# LoRA parameters
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training hyperparameters
lr_stage1 = 5e-5
epochs_stage1 = 3
batch_size_stage1 = 8

lr_stage2 = 1e-5
epochs_stage2 = 5
batch_size_stage2 = 8
early_stop = 2

In [23]:
def run_lora_two_stage_for_rank(r):
    stage1_dir = os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage1_idioms")
    stage2_dir = os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage2_wmt_encfrozen")

    # Stage 1: idioms
    if checkpoint_exists(stage1_dir) and not FORCE_RETRAIN:
        print(f"Loading existing LoRA stage-1 checkpoint from: {stage1_dir}")
    else:
        print(f"Training LoRA stage-1 checkpoint and saving to: {stage1_dir}")
        fine_tune_lora(
            BASE_MODEL,
            idiom_ds["train"],
            idiom_ds["dev"],
            stage1_dir,
            lr=lr_stage1,
            epochs=epochs_stage1,
            batch_size=batch_size_stage1,
            freeze_encoder=False,
            lora_r=r,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            early_stopping_patience=None,
            weight_decay=0.0,
            lora_decoder_only=True,
            adapter_path=None,
        )

    # Stage 2: WMT (encoder frozen), continue adapter
    if checkpoint_exists(stage2_dir) and not FORCE_RETRAIN:
        print(f"Loading existing LoRA stage-2 checkpoint from: {stage2_dir}")
    else:
        print(f"Training LoRA stage-2 checkpoint and saving to: {stage2_dir}")
        fine_tune_lora(
            BASE_MODEL,
            general_ds["ft_train"],
            general_ds["ft_dev"],
            stage2_dir,
            lr=lr_stage2,
            epochs=epochs_stage2,
            batch_size=batch_size_stage2,
            freeze_encoder=True,
            lora_r=r,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            early_stopping_patience=early_stop,
            weight_decay=0.0,
            lora_decoder_only=True,
            adapter_path=stage1_dir,
        )

    return stage1_dir, stage2_dir


rank_runs = {}
for r in LORA_RANKS:
    print(f"\n===== Preparing LoRA rank sweep: r={r} =====")
    s1, s2 = run_lora_two_stage_for_rank(r)
    rank_runs[r] = {"stage1": s1, "stage2": s2}



===== Preparing LoRA rank sweep: r=4 =====
Training LoRA stage-1 checkpoint and saving to: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r4_stage1_idioms


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

trainable params: 196,608 || all params: 74,607,104 || trainable%: 0.2635


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss
1,1.366688,1.306988
2,1.357231,1.285771
3,1.329413,1.280383


Training LoRA stage-2 checkpoint and saving to: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r16_stage2_wmt_encfrozen


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

trainable params: 786,432 || all params: 75,196,928 || trainable%: 1.0458


Epoch,Training Loss,Validation Loss
1,2.229101,2.011804
2,2.110800,1.998784
3,1.836411,1.992118
4,2.156190,1.988738
5,2.048347,1.987707


In [24]:
# check if checkpoints exist
for r in [4, 8, 16]:
    stage2_dir = os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage2_wmt_encfrozen")
    print(f"\n=== r={r} ===")
    print("Exists:", os.path.isdir(stage2_dir))
    if os.path.isdir(stage2_dir):
        print("Files:", os.listdir(stage2_dir))


=== r=4 ===
Exists: True
Files: ['checkpoint-2500', 'checkpoint-5000', 'checkpoint-7500', 'checkpoint-10000', 'checkpoint-12500', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']

=== r=8 ===
Exists: True
Files: ['checkpoint-2500', 'checkpoint-5000', 'checkpoint-7500', 'checkpoint-10000', 'checkpoint-12500', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']

=== r=16 ===
Exists: True
Files: ['checkpoint-2500', 'checkpoint-5000', 'checkpoint-7500', 'checkpoint-10000', 'checkpoint-12500', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']


In [25]:
# create rank runs from checkpoints
LORA_RANKS = [4, 8, 16]

rank_runs = {
    r: {
        "stage1": os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage1_idioms"),
        "stage2": os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage2_wmt_encfrozen"),
    }
    for r in LORA_RANKS
}

# sanity print
for r, paths in rank_runs.items():
    print(f"\n=== r={r} ===")
    print("stage1 exists:", os.path.isdir(paths["stage1"]))
    print("stage2 exists:", os.path.isdir(paths["stage2"]))


=== r=4 ===
stage1 exists: True
stage2 exists: True

=== r=8 ===
stage1 exists: True
stage2 exists: True

=== r=16 ===
stage1 exists: True
stage2 exists: True


In [26]:
def resolve_adapter_dir(stage2_dir):
    # use adapter files in stage2_dir
    if os.path.exists(os.path.join(stage2_dir, "adapter_config.json")):
        return stage2_dir

    # Otherwise, search checkpoint subfolders and pick the latest that has adapter_config.json
    ckpts = [d for d in os.listdir(stage2_dir) if d.startswith("checkpoint-")]
    ckpts = sorted(ckpts, key=lambda x: int(x.split("-")[1]))
    for ck in reversed(ckpts):
        cand = os.path.join(stage2_dir, ck)
        if os.path.exists(os.path.join(cand, "adapter_config.json")):
            return cand

    raise FileNotFoundError(f"No adapter_config.json found in {stage2_dir} or its checkpoint-* subfolders.")

In [27]:
# Decoding sweep
def make_beam_lp_setups(beams=(4, 8), lps=(0.8, 1.0, 1.2)):
    setups = {"greedy": {}}
    for b in beams:
        for lp in lps:
            name = f"beam{b}_lp{lp}"
            setups[name] = {"num_beams": b, "length_penalty": lp, "early_stopping": True}
    return setups

DECODING_SETUPS = make_beam_lp_setups(beams=(4, 8), lps=(0.8, 1.0, 1.2))

In [28]:
def eval_and_log_decoding_sweep(model, tok, run_prefix, model_dir, lr_used, epochs_used, batch_size_used, notes_prefix=""):
    canonical_label = infer_model_label(run_prefix, run_prefix)

    # Idioms
    for decode_name, gen_kwargs in DECODING_SETUPS.items():
        idiom_pred = generate_preds(model, tok, idiom_test["src"], batch_size=16, gen_kwargs=gen_kwargs)
        idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
        print(f"{run_prefix} idioms ({decode_name}):", idiom_m)

        log_metrics(
            f"{run_prefix}_{decode_name}",
            "idioms_test",
            idiom_m,
            model_name=model_dir,
            learning_rate=lr_used,
            epochs=epochs_used,
            batch_size=batch_size_used,
            max_source_len=MAX_SOURCE_LEN,
            max_new_tokens=MAX_NEW_TOKENS,
            freeze_encoder=True,
            train_size=len(general_ds["ft_train"]),
            dev_size=len(general_ds["ft_dev"]),
            seed=SEED,
            n_eval=len(idiom_test),
            notes=f"{notes_prefix} | decoding={decode_name} | {gen_kwargs}",
        )

        if decode_name == CANONICAL_DECODE:
            upsert_canonical_metric(
                canonical_label,
                "idioms_test",
                idiom_m,
                source_notebook="04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb",
                source_run_name=f"{run_prefix}_{decode_name}",
                notes=f"{notes_prefix} | canonical_decode={decode_name}",
            )

    # WMT
    for decode_name, gen_kwargs in DECODING_SETUPS.items():
        wmt_pred = generate_preds(model, tok, wmt_test["src"], batch_size=16, gen_kwargs=gen_kwargs)
        wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
        print(f"{run_prefix} wmt ({decode_name}):", wmt_m)

        log_metrics(
            f"{run_prefix}_{decode_name}",
            "wmt_test",
            wmt_m,
            model_name=model_dir,
            learning_rate=lr_used,
            epochs=epochs_used,
            batch_size=batch_size_used,
            max_source_len=MAX_SOURCE_LEN,
            max_new_tokens=MAX_NEW_TOKENS,
            freeze_encoder=True,
            train_size=len(general_ds["ft_train"]),
            dev_size=len(general_ds["ft_dev"]),
            seed=SEED,
            n_eval=len(wmt_test),
            notes=f"{notes_prefix} | decoding={decode_name} | {gen_kwargs}",
        )

        if decode_name == CANONICAL_DECODE:
            upsert_canonical_metric(
                canonical_label,
                "wmt_test",
                wmt_m,
                source_notebook="04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb",
                source_run_name=f"{run_prefix}_{decode_name}",
                notes=f"{notes_prefix} | canonical_decode={decode_name}",
            )


## Stage 2 Evaluation

In [29]:
# Eval
for r, paths in rank_runs.items():
    stage2_dir = resolve_adapter_dir(paths["stage2"])
    print(f"\n===== Evaluating LoRA r={r} adapter dir: {stage2_dir} =====")

    lora_model, lora_tok = load_lora_for_eval(BASE_MODEL, stage2_dir)

    run_prefix = f"lora_r{r}_two_stage"
    notes_prefix = f"LoRA 2-stage | r={r} | stage2=WMT enc_frozen | stage1=idioms | lr2={lr_stage2}"

    eval_and_log_decoding_sweep(
        lora_model,
        lora_tok,
        run_prefix=run_prefix,
        model_dir=stage2_dir,
        lr_used=lr_stage2,
        epochs_used=epochs_stage2,
        batch_size_used=batch_size_stage2,
        notes_prefix=notes_prefix
    )


===== Evaluating LoRA r=4 adapter dir: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r4_stage2_wmt_encfrozen =====


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

lora_r4_two_stage idioms (greedy): {'bleu': 41.85319732005698, 'chrf': 61.836284051676685}


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_beams', 'early_stopping', 'length_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


lora_r4_two_stage idioms (beam4_lp0.8): {'bleu': 40.83679858111063, 'chrf': 60.81967664354647}
lora_r4_two_stage idioms (beam4_lp1.0): {'bleu': 41.41857366668049, 'chrf': 61.25651872921918}
lora_r4_two_stage idioms (beam4_lp1.2): {'bleu': 41.46970939023122, 'chrf': 61.32338717883289}
lora_r4_two_stage idioms (beam8_lp0.8): {'bleu': 40.76732051665143, 'chrf': 60.689663607052715}
lora_r4_two_stage idioms (beam8_lp1.0): {'bleu': 41.23096694248025, 'chrf': 61.1446161149222}
lora_r4_two_stage idioms (beam8_lp1.2): {'bleu': 41.30673630663333, 'chrf': 61.256632604461124}
lora_r4_two_stage wmt (greedy): {'bleu': 27.568797199400436, 'chrf': 58.493062500108195}
lora_r4_two_stage wmt (beam4_lp0.8): {'bleu': 27.700302717057873, 'chrf': 58.4486082060403}
lora_r4_two_stage wmt (beam4_lp1.0): {'bleu': 27.640945798798572, 'chrf': 58.48830277854813}
lora_r4_two_stage wmt (beam4_lp1.2): {'bleu': 27.552758480146217, 'chrf': 58.490656412662545}
lora_r4_two_stage wmt (beam8_lp0.8): {'bleu': 27.808957323231

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

lora_r8_two_stage idioms (greedy): {'bleu': 42.1787655293237, 'chrf': 61.760908613641895}
lora_r8_two_stage idioms (beam4_lp0.8): {'bleu': 40.82255089546092, 'chrf': 60.79652099136028}
lora_r8_two_stage idioms (beam4_lp1.0): {'bleu': 41.35400706936225, 'chrf': 61.23290268862818}
lora_r8_two_stage idioms (beam4_lp1.2): {'bleu': 41.40149955323913, 'chrf': 61.32406516381892}
lora_r8_two_stage idioms (beam8_lp0.8): {'bleu': 40.8912239909261, 'chrf': 60.852900401198525}
lora_r8_two_stage idioms (beam8_lp1.0): {'bleu': 41.30680023128676, 'chrf': 61.35294432021068}
lora_r8_two_stage idioms (beam8_lp1.2): {'bleu': 41.60098757274608, 'chrf': 61.6074472343421}
lora_r8_two_stage wmt (greedy): {'bleu': 27.4803387263887, 'chrf': 58.45979563899476}
lora_r8_two_stage wmt (beam4_lp0.8): {'bleu': 27.600402953549036, 'chrf': 58.436460387688726}
lora_r8_two_stage wmt (beam4_lp1.0): {'bleu': 27.51667847547613, 'chrf': 58.44003299046878}
lora_r8_two_stage wmt (beam4_lp1.2): {'bleu': 27.47578346508769, 'chr

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

lora_r16_two_stage idioms (greedy): {'bleu': 41.72424543678229, 'chrf': 61.449820290497506}
lora_r16_two_stage idioms (beam4_lp0.8): {'bleu': 40.43789882744096, 'chrf': 60.81102492449115}
lora_r16_two_stage idioms (beam4_lp1.0): {'bleu': 40.68579128911464, 'chrf': 60.942062674724575}
lora_r16_two_stage idioms (beam4_lp1.2): {'bleu': 41.09159687674348, 'chrf': 61.209496654476425}
lora_r16_two_stage idioms (beam8_lp0.8): {'bleu': 40.781720601964174, 'chrf': 60.814745950211815}
lora_r16_two_stage idioms (beam8_lp1.0): {'bleu': 41.06687801234486, 'chrf': 61.23041322962609}
lora_r16_two_stage idioms (beam8_lp1.2): {'bleu': 41.54031696829093, 'chrf': 61.61378570161629}
lora_r16_two_stage wmt (greedy): {'bleu': 27.596422476853355, 'chrf': 58.459373444561535}
lora_r16_two_stage wmt (beam4_lp0.8): {'bleu': 27.72410526208743, 'chrf': 58.42349905037095}
lora_r16_two_stage wmt (beam4_lp1.0): {'bleu': 27.664991558260745, 'chrf': 58.457662774758724}
lora_r16_two_stage wmt (beam4_lp1.2): {'bleu': 27.

### Checkpoint status quick check

In [31]:
paths = {
    "IDIOM_ONLY_DIR": IDIOM_ONLY_DIR if 'IDIOM_ONLY_DIR' in globals() else None,
}

if 'rank_runs' in globals():
    paths.update({f"lora_r{r}_stage1": rank_runs[r]["stage1"] for r in rank_runs})
    paths.update({f"lora_r{r}_stage2": rank_runs[r]["stage2"] for r in rank_runs})

for label, path in paths.items():
    print(label, "->", path)
    if path is not None:
        print("exists:", checkpoint_exists(path))
    print("---")

IDIOM_ONLY_DIR -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/idiom_only_v1
exists: True
---
lora_r4_stage1 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r4_stage1_idioms
exists: True
---
lora_r8_stage1 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r8_stage1_idioms
exists: True
---
lora_r16_stage1 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r16_stage1_idioms
exists: True
---
lora_r4_stage2 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r4_stage2_wmt_encfrozen
exists: True
---
lora_r8_stage2 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r8_stage2_wmt_encfrozen
exists: True
---
lora_r16_stage2 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r16_stage2_wmt_encfrozen
exists: True
---


### Canonical metrics quick check

In [32]:
if os.path.exists(CANONICAL_METRICS_PATH):
    canonical_df = pd.read_csv(CANONICAL_METRICS_PATH)
    display(
        canonical_df[
            canonical_df["model_label"].isin(["lora_r4_stage2", "lora_r8_stage2", "lora_r16_stage2"])
        ].sort_values(["model_label", "split"])
    )
    print("Canonical file:", CANONICAL_METRICS_PATH)
else:
    print("Canonical file not found yet:", CANONICAL_METRICS_PATH)

,timestamp,model,model_label,split,bleu,chrf,source_notebook,source_run_name,notes
4,2026-03-28T23:08:03-07:00,lora_r16_stage2,lora_r16_stage2,idioms_test,41.724245,61.449820,04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb,lora_r16_two_stage_greedy,LoRA 2-stage | r=16 | stage2=WMT enc_frozen | ...
5,2026-03-28T23:11:12-07:00,lora_r16_stage2,lora_r16_stage2,wmt_test,27.596422,58.459373,04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb,lora_r16_two_stage_greedy,LoRA 2-stage | r=16 | stage2=WMT enc_frozen | ...
6,2026-03-28T22:29:15-07:00,lora_r4_stage2,lora_r4_stage2,idioms_test,41.853197,61.836284,04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb,lora_r4_two_stage_greedy,LoRA 2-stage | r=4 | stage2=WMT enc_frozen | s...
7,2026-03-28T22:32:24-07:00,lora_r4_stage2,lora_r4_stage2,wmt_test,27.568797,58.493063,04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb,lora_r4_two_stage_greedy,LoRA 2-stage | r=4 | stage2=WMT enc_frozen | s...
8,2026-03-28T22:48:35-07:00,lora_r8_stage2,lora_r8_stage2,idioms_test,42.178766,61.760909,04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb,lora_r8_two_stage_greedy,LoRA 2-stage | r=8 | stage2=WMT enc_frozen | s...
9,2026-03-28T22:51:45-07:00,lora_r8_stage2,lora_r8_stage2,wmt_test,27.480339,58.459796,04_experiment_idiom_aware_MT_PEFT_LoRA_RANK.ipynb,lora_r8_two_stage_greedy,LoRA 2-stage | r=8 | stage2=WMT enc_frozen | s...


Canonical file: /content/drive/MyDrive/ds266_idiom_mt/results/automatic_metrics_canonical.csv
